In [3]:
!pip -q install evaluate



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.5 MB/s eta 0:00:00


In [ ]:
import torch
import numpy as np
import evaluate
import collections
from datasets import load_dataset
from transformers import GPT2Tokenizer, GPT2ForSequenceClassification, Trainer, TrainingArguments

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)


MODEL_DIR = "./gpt2domain"   #phase2 model

tokenizer = GPT2Tokenizer.from_pretrained(MODEL_DIR)
model = GPT2ForSequenceClassification.from_pretrained(MODEL_DIR, num_labels=2).to(device)

tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id #AI suggested adding this line to avoid warnings about missing pad_token_id.

ds = load_dataset("csv", data_files="classification.csv")["train"]

ds = ds.train_test_split(test_size=0.2, seed=42)
train_ds = ds["train"]
test_ds  = ds["test"]

MAX_LEN = 128

def tok(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN
    )

train_ds = train_ds.map(tok, batched=True)
test_ds  = test_ds.map(tok, batched=True)


train_ds = train_ds.rename_column("label", "labels")
test_ds  = test_ds.rename_column("label", "labels")

train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])


acc = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return acc.compute(predictions=preds, references=labels)

training_args = TrainingArguments(
    output_dir="./gpt2-classifier",
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics
)

trainer.train()

results = trainer.evaluate()
print("Final test results:", results)






device: cuda


Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at ./gpt2domain and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy
1,0.211100,0.000108,1.000000
2,0.000000,0.000021,1.000000
3,0.000000,0.000018,1.000000


Final test results: {'eval_loss': 1.8423796063871123e-05, 'eval_accuracy': 1.0, 'eval_runtime': 0.4595, 'eval_samples_per_second': 217.61, 'eval_steps_per_second': 28.289, 'epoch': 3.0}
